# TAPE LM stage on Colab: fine-tune DeBERTa on the original text and on the GPT explanation

**Saved outputs.** This notebook was run on Colab (one T4 GPU) on 3 and 6 November 2026; this copy does not keep its cell outputs. What it produced is kept: the seven fine-tuned DeBERTa runs (original text, seeds 0-2; original text trained without the test-like holdout, seed 0; GPT explanation, seeds 0-2), with their CLS embeddings, logits and SHA-256 manifest, are in `outputs/tape_lm/` of the released zip (`scripts/download_artifacts.py`). Each run's settings, file size, hash and GPU minutes (42-68 per run) are recorded in `results/shared/embeddings/tape_lm_*.json`. The Stage 2 notebook and the Stage 3 scripts read the embeddings from there.

Reproduces TAPE's `trainLM` for ogbn-products (repo XiaoxinHe/TAPE at d9881f7, `core/LMs/lm_trainer.py`,
`core/LMs/model.py`, `core/config.py`) for the runs in `JOBS`, and saves for every node the 768-d CLS embedding and the
47 logits. The GNNs on top are trained locally with `src/`.

Kept from TAPE: `microsoft/deberta-base`; text built as in TAPE (see `scripts/export_tape_lm_input.py`); max 512
tokens; batch 9; lr 2e-5; 4 epochs; weight decay 0; warmup 900 steps then linear decay to 0; fp16; cross-entropy
with label smoothing 0.3 on the last-layer CLS token; evaluate on val every 1,388 steps and keep the weights with
the lowest val loss; fp16 inference. TAPE's dropout settings are assigned after the model is built under names
DeBERTa does not read, so DeBERTa's default dropout is what TAPE actually trains with; same here.

Different in effect: TAPE's `run.sh` exposes 4 GPUs and the Trainer multiplies the batch by the number of GPUs, so
TAPE's effective batch is 36 with 1,632 steps, of which 900 warm up, and one evaluation. On one T4 we use batch 9:
6,536 steps, 900 of warmup, four evaluations. Batch 36 does not fit on a T4.

Changed: TAPE uses the Hugging Face `Trainer`; this is the same procedure as an explicit loop with the Trainer's
defaults (AdamW betas 0.9/0.999, eps 1e-8, gradient-norm clipping at 1.0, reshuffled every epoch, last partial
batch dropped). Each batch is padded to its own longest sequence instead of the longest in the dataset, and
inference batches are sorted by length: masked padding does not change the result, it only saves time.

Inputs and outputs live on Google Drive, so nothing is lost when the runtime is recycled:
1. Once: put the four `tape_lm_input_part*.jsonl.gz` files (from `outputs/tape_lm/`) in `My Drive/cs5284/tape_lm/`.
2. Runtime > Change runtime type > T4 GPU.
3. Runtime > Run all and allow Drive access. About 2 hours per seed on a T4. Each LM's files are written to
   `My Drive/cs5284/tape_lm/out/` with a SHA-256 manifest (`manifest.json`); check a local copy with
   `scripts/verify_manifest.py`. If the runtime is lost, Run all again: finished LMs are skipped.

In [ ]:
# Colab's transformers 5.x DeBERTa fails under fp16 autocast (masked_fill overflow); pin the version in
# requirements.txt. Run this cell before anything imports transformers.
!pip install -q transformers==4.46.3
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import contextlib
import glob
import gzip
import hashlib
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import transformers
from transformers import AutoModel, AutoTokenizer, DebertaConfig

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/cs5284/tape_lm"
except ImportError:  # local smoke test
    drive, DRIVE_DIR = None, "/content"
INPUT = f"{DRIVE_DIR}/tape_lm_input_part*.jsonl.gz"
OUT = f"{DRIVE_DIR}/out"
MODEL = "microsoft/deberta-base"
# Each job is (text, seed, plan). Plan None trains on the official train nodes, as TAPE does. Otherwise it names
# `{DRIVE_DIR}/{plan}.json` with "exclude" (train nodes to hold out) and "pseudo" ({node id: label}, extra training
# nodes with labels from another model). Seed 0 without a plan finished on 2026-11-03.
JOBS = [("orig", 1, None), ("expl", 1, None), ("orig", 2, None), ("expl", 2, None),
        ("orig", 0, "ex-testlike"), ("orig", 0, "ex-testlike-pl")]
NUM_CLASSES = 47
MAX_LEN = 512
BATCH = 9
EVAL_BATCH = BATCH * 8
LR = 2e-5
SMOKE = False  # True: tiny random model and word-hash tokenizer on CPU, to check the code without a GPU

recs = []
for path in sorted(glob.glob(INPUT)):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        recs += [json.loads(line) for line in f]
assert [r["i"] for r in recs] == list(range(54025)), "missing or extra input parts"
N = len(recs)
train_idx = [r["i"] for r in recs if r["split"] == "train"]
val_idx = [r["i"] for r in recs if r["split"] == "val"]
labels = torch.tensor([r["label"] for r in recs])
print(N, "nodes,", len(train_idx), "train,", len(val_idx), "val")

# TAPE's schedule (lm_trainer.py): step counts are computed from the number of nodes, not of train nodes.
eq_batch = BATCH * 4
EVAL_STEPS = 50000 // eq_batch                    # eval_patience 50000 -> 1388
WARMUP_STEPS = int(0.6 * (N // eq_batch + 1))     # warmup_epochs 0.6 of 1501 -> 900
EPOCHS = 4
INFER_NODES = N
if SMOKE:
    train_idx, val_idx = train_idx[:90], val_idx[:30]
    EVAL_STEPS, WARMUP_STEPS, EPOCHS, INFER_NODES = 4, 2, 2, 300
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
if not SMOKE:
    assert use_amp, "No GPU: Runtime > Change runtime type > T4 GPU"
os.makedirs(OUT, exist_ok=True)
print("eval every", EVAL_STEPS, "steps; warmup", WARMUP_STEPS, "steps;", len(train_idx) // BATCH * EPOCHS, "steps")

In [ ]:
if SMOKE:
    PAD_ID = 0
    def encode(texts):
        return [[1] + [hash(w) % 995 + 5 for w in t.split()][:MAX_LEN - 1] for t in texts]
else:
    tok = AutoTokenizer.from_pretrained(MODEL)
    PAD_ID = tok.pad_token_id
    def encode(texts):
        return tok(texts, truncation=True, max_length=MAX_LEN)["input_ids"]


def make_batch(ids, nodes):
    length = max(len(ids[i]) for i in nodes)
    input_ids = torch.full((len(nodes), length), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(nodes), length), dtype=torch.long)
    for r, i in enumerate(nodes):
        input_ids[r, :len(ids[i])] = torch.tensor(ids[i])
        mask[r, :len(ids[i])] = 1
    return input_ids.to(device), mask.to(device)


class BertClassifier(nn.Module):
    """TAPE's BertClassifier: linear layer on the last-layer CLS token, label-smoothed cross-entropy."""

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, NUM_CLASSES)
        self.loss_func = nn.CrossEntropyLoss(label_smoothing=0.3)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return cls, self.classifier(cls)


def new_encoder():
    if SMOKE:
        return AutoModel.from_config(DebertaConfig(vocab_size=1000, hidden_size=32, num_hidden_layers=2,
                                                   num_attention_heads=2, intermediate_size=64))
    return AutoModel.from_pretrained(MODEL)


def amp():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()


@torch.no_grad()
def evaluate(model, ids, nodes):
    """Mean val loss and accuracy, as the Trainer's evaluation (fp16 autocast, batch 72)."""
    model.eval()
    loss_sum, correct = 0.0, 0
    for b in range(0, len(nodes), EVAL_BATCH):
        chunk = nodes[b:b + EVAL_BATCH]
        with amp():
            _, logits = model(*make_batch(ids, chunk))
        y = labels[chunk].to(device)
        loss_sum += model.loss_func(logits.float(), y).item() * len(chunk)
        correct += (logits.argmax(1) == y).sum().item()
    model.train()
    return loss_sum / len(nodes), correct / len(nodes)

In [ ]:
def training_nodes(plan):
    """The nodes to train on and the labels to train with, for a plan (see JOBS). Val is never touched."""
    if plan is None:
        return train_idx, labels
    with open(f"{DRIVE_DIR}/{plan}.json") as f:
        spec = json.load(f)
    exclude, pseudo = set(spec["exclude"]), {int(k): v for k, v in spec["pseudo"].items()}
    assert not (set(val_idx) & (exclude | set(pseudo))), "val nodes must stay out of training"
    assert not (set(train_idx) - exclude) & set(pseudo), "a labelled training node also got a pseudo-label"
    y = labels.clone()
    if pseudo:
        y[list(pseudo)] = torch.tensor(list(pseudo.values()))
    return [i for i in train_idx if i not in exclude] + sorted(pseudo), y


def run(text, seed, plan):
    t_start = time.time()
    ids = encode([r[text] for r in recs])
    fit_idx, fit_labels = training_nodes(plan)
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = BertClassifier(new_encoder()).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)
    steps_per_epoch = len(fit_idx) // BATCH  # last partial batch dropped
    total_steps = steps_per_epoch * EPOCHS
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(1, WARMUP_STEPS) if s < WARMUP_STEPS
        else max(0.0, (total_steps - s) / max(1, total_steps - WARMUP_STEPS)))
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    history, best_loss, best_step, best_state = [], math.inf, None, None
    step, run_loss = 0, 0.0
    model.train()
    for epoch in range(EPOCHS):
        perm = torch.randperm(len(fit_idx), generator=gen).tolist()
        for b in range(steps_per_epoch):
            nodes = [fit_idx[k] for k in perm[b * BATCH:(b + 1) * BATCH]]
            with amp():
                _, logits = model(*make_batch(ids, nodes))
                loss = model.loss_func(logits.float(), fit_labels[nodes].to(device))
            opt.zero_grad(set_to_none=True)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            step += 1
            run_loss += loss.item()
            if step % 100 == 0:
                history.append({"step": step, "train_loss": run_loss / 100, "lr": sched.get_last_lr()[0]})
                run_loss = 0.0
            if step % EVAL_STEPS == 0:
                val_loss, val_acc = evaluate(model, ids, val_idx)
                history.append({"step": step, "val_loss": val_loss, "val_acc": val_acc})
                print(f"{text} step {step}/{total_steps}: val loss {val_loss:.4f}, val acc {val_acc:.4f}, "
                      f"{(time.time() - t_start) / 60:.1f} min")
                if val_loss < best_loss:
                    best_loss, best_step = val_loss, step
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:  # as load_best_model_at_end: the last steps after the final eval are dropped
        model.load_state_dict(best_state)
    t_train = time.time() - t_start

    # Inference on every node, fp16 as TAPE's fp16_full_eval; batches sorted by length.
    model.eval()
    if use_amp:
        model.half()
    order = sorted(range(INFER_NODES), key=lambda i: len(ids[i]))
    emb = np.zeros((N, model.encoder.config.hidden_size), dtype=np.float16)
    pred = np.zeros((N, NUM_CLASSES), dtype=np.float16)
    with torch.no_grad():
        for b in range(0, len(order), EVAL_BATCH):
            chunk = order[b:b + EVAL_BATCH]
            cls, logits = model(*make_batch(ids, chunk))
            emb[chunk] = cls.float().cpu().numpy()
            pred[chunk] = logits.float().cpu().numpy()
    t_total = time.time() - t_start

    name = output_name(text, seed, plan)
    np.save(f"{OUT}/{name}.emb.npy", emb)
    np.save(f"{OUT}/{name}.pred.npy", pred)
    y = labels.numpy()
    gold_idx = sorted(set(train_idx) & set(fit_idx))  # official train nodes it actually trained on
    tr_acc = float((pred[gold_idx].argmax(1) == y[gold_idx]).mean())
    va_acc = float((pred[val_idx].argmax(1) == y[val_idx]).mean())
    meta = {
        "text": text, "model": MODEL, "seed": seed, "plan": plan, "fit_nodes": len(fit_idx),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name() if use_amp else "cpu",
        "batch": BATCH, "lr": LR, "epochs": EPOCHS, "total_steps": total_steps, "eval_steps": EVAL_STEPS,
        "warmup_steps": WARMUP_STEPS, "best_step": best_step, "best_val_loss": best_loss,
        "train_acc_fp16_inference": tr_acc, "val_acc_fp16_inference": va_acc,
        "train_minutes": t_train / 60, "total_minutes": t_total / 60, "history": history,
    }
    with open(f"{OUT}/{name}.json", "w") as f:
        json.dump(meta, f, indent=1)
    print(f"{name}: best step {best_step}, train acc {tr_acc:.4f}, val acc {va_acc:.4f}, {t_total / 60:.1f} min")
    del model, opt, best_state
    if use_amp:
        torch.cuda.empty_cache()


def output_name(text, seed, plan):
    return f"tape_lm_{text}_deberta-base_seed{seed}" + (f"_{plan}" if plan else "")


def write_manifest():
    """Bytes and SHA-256 of every output file, to check the downloaded copies against."""
    files = {}
    for fname in sorted(os.listdir(OUT)):
        if fname != "manifest.json":
            with open(f"{OUT}/{fname}", "rb") as f:
                files[fname] = {"bytes": os.path.getsize(f"{OUT}/{fname}"), "sha256": hashlib.sha256(f.read()).hexdigest()}
    with open(f"{OUT}/manifest.json", "w") as f:
        json.dump(files, f, indent=1)


for text, seed, plan in JOBS:
    if os.path.exists(f"{OUT}/{output_name(text, seed, plan)}.json"):  # the .json is written last, so this LM finished
        print(output_name(text, seed, plan), "already done, skipped")
        continue
    run(text, seed, plan)
    write_manifest()
print(json.dumps(json.load(open(f"{OUT}/manifest.json")), indent=1))
if drive is not None:
    drive.flush_and_unmount()  # make sure everything has reached Drive before the runtime can go away